# Edge3D Single-Sample Inference

This notebook runs single-sample Edge3D x0-bridge inference from one NPZ file and exports all comparison outputs as PLY.

Current default runtime values come from config:
- `inference.num_inference_steps`, falling back to `validation.num_inference_steps`
- `inference.cfg_scale`, falling back to `validation.cfg_scale`
- `inference.inference_dtype`, falling back to `validation.inference_dtype`

In the current experiment configs, the defaults are `20` inference steps and `1.5` CFG.

The notebook intentionally reuses the same shared implementation as `inference.py`, so notebook and CLI inference stay aligned.

## Workflow

1. Define the inference config in one place.
2. Resolve `num_inference_steps`, `cfg_scale`, and `inference_dtype` from config.
3. Inspect the NPZ condition and ground-truth tensors before running the model.
4. Resolve the checkpoint path, including DeepSpeed stage-2 `last.ckpt/` directories.
5. Run generation with configurable step count and CFG.
6. Compute prediction losses against ground truth.
7. Export `model`, `prediction`, `ground truth`, and merged comparison point clouds as PLY.
8. Print the runtime parameters actually used for this inference.
9. Reuse the same shared Python entrypoint as `inference.py`.

In [ ]:
from pathlib import Path

CONFIG_PATH = Path("configs/experiment/edge3d_flow_train.yaml")
SAMPLE_PATH = Path("/absolute/path/to/sample.npz")
OUTPUT_DIR = Path("demo_outputs/inference_single_sample")
CHECKPOINT = "auto"
DEVICE = "cuda"
NUM_STEPS = None
CFG_SCALE = None
INFERENCE_DTYPE = None

In [ ]:
from pprint import pprint

import numpy as np

from edge3d.tensor_format import load_sample_modalities
from tools.run_edge3d_x0_bridge_inference import (
    resolve_checkpoint_path,
    resolve_inference_settings,
    run_single_sample_inference,
)
from utils import load_yaml_config

config = load_yaml_config(CONFIG_PATH)
runtime = resolve_inference_settings(
    config,
    num_steps=NUM_STEPS,
    cfg_scale=CFG_SCALE,
    inference_dtype=INFERENCE_DTYPE,
)
checkpoint_path = resolve_checkpoint_path(CHECKPOINT, config)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"config: {CONFIG_PATH}")
print(f"sample: {SAMPLE_PATH}")
print(f"checkpoint: {checkpoint_path}")
print(f"output_dir: {OUTPUT_DIR.resolve()}")
print(f"num_steps: {runtime['num_steps']}")
print(f"cfg_scale: {runtime['cfg_scale']}")
print(f"inference_dtype: {runtime['inference_dtype']}")

In [ ]:
payload = load_sample_modalities(SAMPLE_PATH)
required_keys = ("model_rgb", "model_depth", "model_normal", "edge_depth")

for key in required_keys:
    if key not in payload:
        raise KeyError(f"Missing required key in NPZ payload: {key}")
    array = np.asarray(payload[key])
    finite = array[np.isfinite(array)]
    value_range = (float(finite.min()), float(finite.max())) if finite.size > 0 else (None, None)
    print(
        f"{key}: shape={array.shape}, dtype={array.dtype}, finite_range={value_range}"
    )

print(f"uid: {payload['uid']}")

In [ ]:
RESULT = run_single_sample_inference(
    CONFIG_PATH,
    sample_path=SAMPLE_PATH,
    output_dir=OUTPUT_DIR,
    checkpoint=CHECKPOINT,
    device=DEVICE,
    num_steps=NUM_STEPS,
    cfg_scale=CFG_SCALE,
    inference_dtype=INFERENCE_DTYPE,
)

print(f"num_steps={RESULT['num_steps']}")
print(f"cfg_scale={RESULT['cfg_scale']}")
print(f"uses_cfg={RESULT['uses_cfg']}")
print(f"condition_channels={RESULT['condition_channels']}")
print(f"effective_inference_dtype={RESULT['effective_inference_dtype']}")
pprint(RESULT['losses'])
RESULT

In [ ]:
print("PLY outputs:")
for name, path in sorted(RESULT["pointclouds"].items()):
    print(f"- {name}: {path}")